# TTC Subway Delay Data — Final Cleaning

## Overview
This notebook loads the cleaned TTC subway delay dataset and performs final preparation for analysis.

## Data Source
Raw delay data was sourced from the [City of Toronto Open Data Portal](https://open.toronto.ca/) for years 2021–2025.

## Prior Cleaning (Excel)
Initial cleaning was performed in Excel before loading this notebook:
- Filtered to **significant delays only** (delays ≥ 5 minutes)
- Standardized station names to official TTC station names (lowercase)
- Added a `month` column derived from the `date` field
- Removed non-station entries and noise records

The output of that process is saved at:  
`data/processed/ttc_significant_delay_clean_station.csv`

## This Notebook
Loads the cleaned CSV and continues with Python-based cleaning and validation.

In [2]:
import pandas as pd
import os
import sys

In [3]:
ttc = pd.read_csv("../../data/processed/final_for_EDA/ttc_significant_delay_clean_station.csv",
                  low_memory=False)

print(ttc.shape)
ttc.head()

(21367, 12)


,year,month,date,time,day,station,code,min_delay,min_gap,bound,line,vehicle
0,2025,March,2025-03-16,13:49,Sunday,old mill,PUTIJ,10,14,E,BD,5103
1,2025,January,2025-01-01,2:10,Wednesday,bathurst,MUSAN,5,9,E,BD,5227
2,2025,February,2025-02-05,20:38,Wednesday,bathurst,SUDP,5,9,W,BD,5148
3,2025,March,2025-03-21,11:40,Friday,bathurst,MUDD,5,9,W,BD,5170
4,2025,June,2025-06-11,15:32,Wednesday,bathurst,SUO,5,9,E,BD,5035


In [4]:
ttc.dtypes

year          int64
month        object
date         object
time         object
day          object
station      object
code         object
min_delay     int64
min_gap       int64
bound        object
line         object
vehicle       int64
dtype: object

In [5]:
ttc.isnull().sum()

year           0
month          0
date           0
time           0
day            0
station        0
code           0
min_delay      0
min_gap        0
bound        251
line           1
vehicle        0
dtype: int64

## Clean the `line` Column

Inspection revealed several invalid entries:
- **Remapped:** `YUS` → `YU`, `BD LINE 2` → `BD`
- **Dropped:** `SRT` (Line 3, closed 2023), `YU/BD` (ambiguous interchange), and bus routes (`109 RANEE`, `20 CLIFFSIDE`, `77 SWANSEA`) which are data entry errors

In [6]:
# Remap variations to standard line names
ttc['line'] = ttc['line'].replace({
    'YUS':       'YU',
    'BD LINE 2': 'BD'
})

# Keep only valid subway lines (drops SRT, YU/BD, and bus routes)
valid_lines = ['YU', 'BD', 'SHP']
ttc = ttc[ttc['line'].isin(valid_lines)]

# Verify
print(ttc.shape)
ttc['line'].value_counts()

(20613, 12)


line
YU     12350
BD      7300
SHP      963
Name: count, dtype: int64

In [7]:
# Inspect missing bound values
ttc[ttc['bound'].isnull()][['station', 'line', 'bound']].value_counts()


Series([], Name: count, dtype: int64)

In [8]:
# Inspect the missing line value
ttc[ttc['line'].isnull()]

,year,month,date,time,day,station,code,min_delay,min_gap,bound,line,vehicle


In [9]:
ttc['line'].value_counts()


line
YU     12350
BD      7300
SHP      963
Name: count, dtype: int64

## Clean the `bound` Column

The `bound` column records the direction of travel (N, S, E, W) for each delay incident.

In the raw data, many `bound` values were missing. During the Excel cleaning phase, missing values were **manually imputed based on station** — for example, a station on a one-directional segment could only have a limited set of valid directions. This reduced the missing values significantly.

The remaining 251 NaN values could not be reliably determined and are dropped here, representing ~1.2% of the dataset.

In [10]:
# Drop remaining rows with missing bound values
ttc = ttc[ttc['bound'].notna()]

# Verify no missing values remain
print(ttc.shape)
ttc.isnull().sum()

(20396, 12)


year         0
month        0
date         0
time         0
day          0
station      0
code         0
min_delay    0
min_gap      0
bound        0
line         0
vehicle      0
dtype: int64

In [11]:
# Inspect station column
print("Unique station count:", ttc['station'].nunique())
print()
ttc['station'].value_counts()

Unique station count: 97



station
bloor                    968
eglinton                 816
st george                769
finch                    651
kipling                  612
                        ... 
warden                     1
VIADUCT                    1
 york mills                1
sheppard                   1
YONGE UNIVERSITY LINE      1
Name: count, Length: 97, dtype: int64

## Clean the `station` Column

Station names required additional normalization after Excel cleaning:
- Stripped leading/trailing whitespace (caused duplicate entries e.g. `york mills`)
- Lowercased all entries for consistency
- Remapped descriptive/route entries to official station names
- Dropped non-station entries (maintenance yards, infrastructure labels, route descriptions)

In [12]:
# Step 1: Strip whitespace and lowercase
ttc['station'] = ttc['station'].str.strip().str.lower()

# Step 2: Remap variations to official station names
ttc['station'] = ttc['station'].replace({
    'sheppard to eglinton':    'sheppard-yonge',
    'sheppard to  york mills': 'sheppard-yonge',
    'sheppard':                'sheppard-yonge',
    'sheppard_yonge':          'sheppard-yonge',   # fix underscore
    'bloor-danforth line':     'bloor',
    'line 1 bloor to queen':   'bloor',
    'finch to union':          'finch',
    'clencarin':               'glencairn',
    'glencarin':               'glencairn',
    'lasndowne':               'lansdowne',
    'greenwood wye':           'greenwood',
    'st patrick to':           'st patrick',
    'main':                    'main street',
    'pioneer village':         'highway 407',      # original construction name before station opened
})

# Step 3: Drop non-station entries (yards, infrastructure, route descriptions)
drop_stations = [
    'yonge university line',
    'viaduct',
    'massey creek ee',
    'hostler 2 wilson yard',
    'kipling tail track 2',
    'kipling tail track',
    'south hostler',
    'sheppard line',
    'finch tail',
    'greenwood portal',
    'eglinton yard',
    'keele yard',
    'greenwood yard',
    'wilson yard',
    'wilson south holster',
    'tmu',
    'cedarvale',               # not an official TTC subway station
]
ttc = ttc[~ttc['station'].isin(drop_stations)]

# Verify
print("Unique station count:", ttc['station'].nunique())
print("Shape:", ttc.shape)
ttc['station'].value_counts().reset_index()

Unique station count: 69
Shape: (20307, 12)


,station,count
0,bloor,970
1,eglinton,816
2,st george,769
3,finch,652
4,kipling,612
...,...,...
64,old mill,127
65,bayview,112
66,runnymede,92
67,york university,86


## Check for Duplicates and Outliers

In [13]:
# Check for duplicate rows
print("Duplicate rows:", ttc.duplicated().sum())
ttc[ttc.duplicated(keep=False)].sort_values(['date', 'time', 'station'])

Duplicate rows: 3


,year,month,date,time,day,station,code,min_delay,min_gap,bound,line,vehicle
19940,2023,September,2023-09-18,5:55,Monday,wilson,PUTDN,33,0,S,YU,6081
19941,2023,September,2023-09-18,5:55,Monday,wilson,PUTDN,33,0,S,YU,6081
12095,2024,June,2024-06-28,1:43,Friday,ossington,SUDP,5,10,W,BD,5355
12096,2024,June,2024-06-28,1:43,Friday,ossington,SUDP,5,10,W,BD,5355
16704,2025,December,2025-12-17,16:13,Wednesday,st george,SUDP,5,8,N,YU,6061
16705,2025,December,2025-12-17,16:13,Wednesday,st george,SUDP,5,8,N,YU,6061


In [14]:
# Drop duplicate rows, keeping the first occurrence
ttc = ttc.drop_duplicates()

print("Duplicates remaining:", ttc.duplicated().sum())
print("Shape after dropping duplicates:", ttc.shape)

Duplicates remaining: 0
Shape after dropping duplicates: (20304, 12)


In [15]:
# Check numeric columns for outliers
ttc[['min_delay', 'min_gap']].describe()

,min_delay,min_gap
count,20304.000000,20304.000000
mean,11.438288,14.839046
std,22.546119,20.695914
min,5.000000,0.000000
25%,5.000000,10.000000
50%,7.000000,11.000000
75%,11.000000,15.000000
max,900.000000,906.000000


In [16]:
# How many rows exceed common thresholds?
for mins in [60, 120, 180, 360]:
    count = (ttc['min_delay'] > mins).sum()
    print(f"min_delay > {mins} min: {count} rows")


min_delay > 60 min: 267 rows
min_delay > 120 min: 95 rows
min_delay > 180 min: 40 rows
min_delay > 360 min: 16 rows


In [16]:
# See the actual extreme values
ttc[ttc['min_delay'] > 60][['date', 'station', 'line', 'code', 'min_delay', 'min_gap']]\
    .sort_values('min_delay', ascending=False)


,date,station,line,code,min_delay,min_gap
14096,2025-02-16,sheppard west,YU,MUWEA,900,906
20123,2026-01-26,woodbine,BD,PUTIS,827,0
4817,2025-02-16,eglinton,YU,MUWEA,807,812
1115,2024-05-13,broadview,BD,EUCD,716,719
15588,2026-01-26,st clair,YU,PUTIS,708,0
...,...,...,...,...,...,...
20704,2025-05-02,york mills,YU,SUUT,61,67
14510,2024-01-14,sheppard-yonge,SHP,PUTWZ,61,0
16895,2024-04-10,st george,YU,SUUT,61,66
18682,2023-10-07,victoria park,BD,MUI,61,67


In [17]:
# Inspect extreme delay values
ttc[ttc['min_delay'] > 120][['date', 'station', 'line', 'code', 'min_delay', 'min_gap']]\
    .sort_values('min_delay', ascending=False)

,date,station,line,code,min_delay,min_gap
14096,2025-02-16,sheppard west,YU,MUWEA,900,906
20123,2026-01-26,woodbine,BD,PUTIS,827,0
4817,2025-02-16,eglinton,YU,MUWEA,807,812
1115,2024-05-13,broadview,BD,EUCD,716,719
15588,2026-01-26,st clair,YU,PUTIS,708,0
...,...,...,...,...,...,...
20497,2023-06-28,bloor,BD,MUPR1,125,133
18384,2022-05-28,vaughan metropolitan centre,YU,SUBT,125,131
17540,2023-03-06,summerhill,YU,MUPR1,124,127
7017,2022-06-17,greenwood,BD,MUPR1,122,129


## Remove Outliers — `min_delay`

Per the project README: *"Remove delays greater than 60 minutes to focus on realistic operational scenarios."*

`min_delay` is a **feature** (not the target) used to help classify the cause of delay (`code`). Delays over 60 minutes are rare edge cases that may represent data entry errors or exceptional one-off events unlikely to generalise in a classification model.

In [18]:
# Drop delays over 60 minutes
ttc = ttc[ttc['min_delay'] <= 60]

# Verify
print("Shape:", ttc.shape)
ttc[['min_delay', 'min_gap']].describe()

Shape: (20037, 12)


,min_delay,min_gap
count,20037.000000,20037.00000
mean,9.710536,13.31881
std,7.357650,7.91312
min,5.000000,0.00000
25%,5.000000,10.00000
50%,7.000000,11.00000
75%,11.000000,15.00000
max,60.000000,70.00000


## Prepare Data Types
Converting `date` to `datetime64` to enable proper date operations.

> **TODO (group review):** Decide on handling for `time`, `month`, and `day` columns:
> - `time` (object, e.g. `"13:49"`) — extract `hour` as int for ML, or keep raw?
> - `month` / `day` (object) — convert to ordered `category`, integer, or leave as-is?

In [19]:
# Convert date to datetime
ttc['date'] = pd.to_datetime(ttc['date'])


# Ordered categoricals for month and day
ttc['month'] = pd.Categorical(ttc['month'], categories=[
    'January','February','March','April','May','June',
    'July','August','September','October','November','December'], ordered=True)

ttc['day'] = pd.Categorical(ttc['day'], categories=[
    'Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'], ordered=True)

print(ttc.dtypes)


year                  int64
month              category
date         datetime64[ns]
time                 object
day                category
station              object
code                 object
min_delay             int64
min_gap               int64
bound                object
line                 object
vehicle               int64
dtype: object


## Save Cleaned Dataset

In [20]:
# Save final cleaned dataset for EDA and modelling
output_path = "../../data/processed/ttc_cleaned_final.csv"
ttc.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Final shape: {ttc.shape}")
ttc.head()

Saved: ../../data/processed/ttc_cleaned_final.csv
Final shape: (20037, 12)


,year,month,date,time,day,station,code,min_delay,min_gap,bound,line,vehicle
0,2025,March,2025-03-16,13:49,Sunday,old mill,PUTIJ,10,14,E,BD,5103
1,2025,January,2025-01-01,2:10,Wednesday,bathurst,MUSAN,5,9,E,BD,5227
2,2025,February,2025-02-05,20:38,Wednesday,bathurst,SUDP,5,9,W,BD,5148
3,2025,March,2025-03-21,11:40,Friday,bathurst,MUDD,5,9,W,BD,5170
4,2025,June,2025-06-11,15:32,Wednesday,bathurst,SUO,5,9,E,BD,5035
